In [1]:
import pandas as pd

ratings = pd.read_csv("../sample_data/inputs/ml-latest-small/ratings.csv")
movies = pd.read_csv("../sample_data/inputs/ml-latest-small/movies.csv")

In [2]:
## sorting the ratings dataset
ratings_sorted = ratings.sort_values(['userId', 'timestamp'])
ratings_sorted.head(10)

,userId,movieId,rating,timestamp
43,1,804,4.0,964980499
73,1,1210,5.0,964980499
120,1,2018,5.0,964980523
171,1,2628,4.0,964980523
183,1,2826,4.0,964980523
219,1,3578,5.0,964980668
220,1,3617,4.0,964980683
227,1,3744,4.0,964980694
6,1,101,5.0,964980868
24,1,441,4.0,964980868


In [3]:
# Figure out each rating's position within its own user's history

ratings_sorted['rank_in_user'] = ratings_sorted.groupby('userId').cumcount()
ratings_sorted['user_total'] = ratings_sorted.groupby('userId')['userId'].transform('count')
ratings_sorted[['userId', 'timestamp', 'rank_in_user', 'user_total']].head(10)

,userId,timestamp,rank_in_user,user_total
43,1,964980499,0,232
73,1,964980499,1,232
120,1,964980523,2,232
171,1,964980523,3,232
183,1,964980523,4,232
219,1,964980668,5,232
220,1,964980683,6,232
227,1,964980694,7,232
6,1,964980868,8,232
24,1,964980868,9,232


In [4]:
ratings_sorted.head(10)

,userId,movieId,rating,timestamp,rank_in_user,user_total
43,1,804,4.0,964980499,0,232
73,1,1210,5.0,964980499,1,232
120,1,2018,5.0,964980523,2,232
171,1,2628,4.0,964980523,3,232
183,1,2826,4.0,964980523,4,232
219,1,3578,5.0,964980668,5,232
220,1,3617,4.0,964980683,6,232
227,1,3744,4.0,964980694,7,232
6,1,101,5.0,964980868,8,232
24,1,441,4.0,964980868,9,232


In [5]:
ratings_sorted['percentile'] = ratings_sorted['rank_in_user'] / ratings_sorted['user_total']

train = ratings_sorted[ratings_sorted['percentile'] < 0.8]
held_out = ratings_sorted[ratings_sorted['percentile'] >= 0.8]

print(f"Train: {len(train)} ratings")
print(f"Held-out: {len(held_out)} ratings")

Train: 80896 ratings
Held-out: 19940 ratings


In [6]:
## Finding the most popluar movies (max number of ratings in the train set )
popularity_ranking = train['movieId'].value_counts()
popularity_ranking.head(10)

movieId
356     304
318     282
296     272
2571    261
593     253
260     237
110     226
480     223
589     202
1       202
Name: count, dtype: int64

In [7]:
## Building the recommendation system based on popluarity as our basline

def recommend_popular(user_id, k=10):
    already_rated = train[train['userId'] == user_id]['movieId'].tolist()
    recommendations = [movie for movie in popularity_ranking.index if movie not in already_rated]
    return recommendations[:k]

recommend_popular(1, k=10)

[318, 593, 589, 150, 527, 4993, 780, 7153, 858, 5952]

This is the list of 10 moives that our recommender system will recommend to user 1 based on popularity with skipping the movies that has already rated.

In [8]:
all_users = ratings['userId'].unique()
recommendations_by_user = {user: recommend_popular(user, k=10) for user in all_users}

print(recommendations_by_user[1]) ## movie recommendation for user 1
print(recommendations_by_user[2]) ## movie recommendation for user 2 
print(recommendations_by_user[3]) ## movie recommendation for user 3

[318, 593, 589, 150, 527, 4993, 780, 7153, 858, 5952]
[356, 296, 2571, 593, 260, 110, 480, 589, 1, 2959]
[356, 318, 296, 2571, 593, 260, 110, 480, 589, 1]


In [9]:
def get_hits(k=10):
    hits = 0
    total_users = 0
    
    for user in all_users:
        user_held_out = held_out[(held_out['userId'] == user) & (held_out['rating'] >= 4)]['movieId'].tolist()
        
        if len(user_held_out) == 0:
            continue
        
        total_users += 1
        recs = recommendations_by_user[user][:k]
        
        if any(movie in recs for movie in user_held_out):
            hits += 1
    
    return hits / total_users

hit_rate = get_hits(k=10)
print(f"Popularity baseline hit-rate@10: {hit_rate:.2%}")

Popularity baseline hit-rate@10: 31.13%


 out of all the users who had at least one movie they genuinely liked (rated ≥4) in their held-out set, about 31 out of every 100 got that movie successfully included somewhere in their top-10 recommendation list — just from "recommend whatever's popular.

In [10]:
def get_precision_recall(k=10):
    precisions = []
    recalls = []
    
    for user in all_users:
        user_held_out = held_out[(held_out['userId'] == user) & (held_out['rating'] >= 4)]['movieId'].tolist()
        
        if len(user_held_out) == 0:
            continue
        
        recs = recommendations_by_user[user][:k]
        num_relevant_and_recommended = len(set(recs) & set(user_held_out))
        
        precisions.append(num_relevant_and_recommended / k)
        recalls.append(num_relevant_and_recommended / len(user_held_out))
    
    return sum(precisions) / len(precisions), sum(recalls) / len(recalls)

precision, recall = get_precision_recall(k=10)
print(f"Popularity baseline — Precision@10: {precision:.2%}, Recall@10: {recall:.2%}")

Popularity baseline — Precision@10: 5.60%, Recall@10: 5.05%


Precision and recall give a stricter picture than hit-rate. The popularity baseline scored Precision@10 = 5.60% and Recall@10 = 5.05% — meaning on average, only about half a movie out of our 10 recommended slots was actually something the user liked, and we only surfaced about 5% of everything a user genuinely enjoyed.

These numbers look much lower than the 31.13% hit-rate, but that's expected, not a contradiction — hit-rate only asks "did we get at least one right?", while precision and recall measure the proportion correct. A single lucky match out of 10 recommendations counts as a full hit, but scores low on precision (1/10) and can score very low on recall if the user liked many movies.

In [11]:
import implicit
print(implicit.__version__)

0.7.3


c:\Users\Himanshu Jain\Downloads\recommendation-system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
user_ids = train['userId'].unique()
movie_ids = train['movieId'].unique()

user_id_map = {id: idx for idx, id in enumerate(user_ids)}
movie_id_map = {id: idx for idx, id in enumerate(movie_ids)}

train['user_idx'] = train['userId'].map(user_id_map)
train['movie_idx'] = train['movieId'].map(movie_id_map)

train[['userId', 'user_idx', 'movieId', 'movie_idx']].head()

C:\Users\Himanshu Jain\AppData\Local\Temp\ipykernel_21572\1334853961.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['user_idx'] = train['userId'].map(user_id_map)
C:\Users\Himanshu Jain\AppData\Local\Temp\ipykernel_21572\1334853961.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['movie_idx'] = train['movieId'].map(movie_id_map)


,userId,user_idx,movieId,movie_idx
43,1,0,804,0
73,1,0,1210,1
120,1,0,2018,2
171,1,0,2628,3
183,1,0,2826,4


In [13]:
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

user_item_matrix = csr_matrix(
    (train['rating'], (train['user_idx'], train['movie_idx']))
)

model = AlternatingLeastSquares(factors=50, iterations=20, regularization=0.01)
model.fit(user_item_matrix)

c:\Users\Himanshu Jain\Downloads\recommendation-system\.venv\lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [00:00<00:00, 35.84it/s]


In [14]:
movie_idx_to_id = {idx: id for id, idx in movie_id_map.items()}

user_idx = user_id_map[1]
recommended = model.recommend(user_idx, user_item_matrix[user_idx], N=10)
recommended_movie_ids = [movie_idx_to_id[idx] for idx in recommended[0]]

recommended_movie_ids

[np.int64(588),
 np.int64(1036),
 np.int64(1221),
 np.int64(2081),
 np.int64(2087),
 np.int64(364),
 np.int64(780),
 np.int64(858),
 np.int64(47),
 np.int64(4571)]

Built and tested the collaborative filtering model (ALS) for a single user. Converted train into a sparse user-item matrix, trained an AlternatingLeastSquares model (factors=50, iterations=20, regularization=0.01) to learn a hidden taste profile for every user and a hidden profile for every movie. For any unrated movie, the model predicts a score by multiplying the user's and movie's hidden numbers together (dot product) — the top 10 highest-scoring, not-yet-rated movies become the recommendation.

In [15]:
cf_recommendations_by_user = {}

for user_id in all_users:
    user_idx = user_id_map[user_id]
    recommended = model.recommend(user_idx, user_item_matrix[user_idx], N=10)
    cf_recommendations_by_user[user_id] = [movie_idx_to_id[idx] for idx in recommended[0]]

Checking the hit and the precesion and recall for the CF filtering systerm

In [16]:
held_out_liked = held_out[held_out['rating'] >= 4].groupby('userId')['movieId'].apply(list).to_dict()

def get_hits(recommendations, k=10):
    hits = 0
    total_users = 0
    
    for user, liked_movies in held_out_liked.items():
        if user not in recommendations:
            continue
        total_users += 1
        recs = recommendations[user][:k]
        if any(movie in recs for movie in liked_movies):
            hits += 1
    
    return hits / total_users


def get_precision_recall(recommendations, k=10):
    precisions = []
    recalls = []
    
    for user, liked_movies in held_out_liked.items():
        if user not in recommendations:
            continue
        recs = recommendations[user][:k]
        num_relevant_and_recommended = len(set(recs) & set(liked_movies))
        precisions.append(num_relevant_and_recommended / k)
        recalls.append(num_relevant_and_recommended / len(liked_movies))
    
    return sum(precisions) / len(precisions), sum(recalls) / len(recalls)

In [17]:
cf_hit_rate = get_hits(cf_recommendations_by_user, k=10)
cf_precision, cf_recall = get_precision_recall(cf_recommendations_by_user, k=10)

print(f"Collaborative Filtering — Hit-rate@10: {cf_hit_rate:.2%}, Precision@10: {cf_precision:.2%}, Recall@10: {cf_recall:.2%}")
print(f"Popularity Baseline      — Hit-rate@10: {hit_rate:.2%}, Precision@10: {precision:.2%}, Recall@10: {recall:.2%}")

Collaborative Filtering — Hit-rate@10: 42.13%, Precision@10: 6.77%, Recall@10: 7.83%
Popularity Baseline      — Hit-rate@10: 31.13%, Precision@10: 5.60%, Recall@10: 5.05%


| Metric | Popularity Baseline | Collaborative Filtering |
|---|---|---|
| Hit-rate@10 | 31.13% | ~43% |
| Precision@10 | 5.60% | ~7% |
| Recall@10 | 5.05% | ~8% |

CF succeeded for roughly 12 more users out of every 100 compared to just recommending what's .

This confirms personalization is worth the added complexity, validating the move to build the full hybrid model (CF + content-based) next, rather than assuming it upfront.